# Flow Control and Multi-Tenant Fairness

In production, you rarely serve a single user or a single workload. Multiple
teams, applications, and priority levels compete for the same inference capacity.
Without flow control, a single heavy user can starve everyone else.

llm-d's **flow control** system provides:
- **Priority bands** to ensure critical requests are served first
- **Fairness policies** to distribute capacity equitably across tenants
- **Late-binding scheduling** to maximize utilization while respecting SLOs

This notebook walks through configuring and testing these features.

## The 3-Tier Dispatch Hierarchy

llm-d's flow control uses a three-tier hierarchy to decide which request
gets served next:

### Tier 1: Priority
Higher-priority requests are always dispatched before lower-priority ones.
Priority bands are defined by the operator (e.g., interactive=100, batch=50,
background=10). Requests inherit their priority from HTTP headers or API keys.

### Tier 2: Fairness
Within the same priority band, requests are distributed fairly across tenants.
Fairness policies can be round-robin (equal share), weighted (proportional to
allocation), or strict (first-come-first-served). Tenants are identified by
a configurable header (e.g., `x-tenant-id`).

### Tier 3: Ordering
Within the same priority band and tenant, requests are ordered by arrival time
(FIFO). This ensures that individual tenants experience predictable latency.

```
Incoming Requests
       |
       v
+------------------+
| Priority Filter  |  "Is this request high or low priority?"
+--------+---------+
         |
         v
+------------------+
| Fairness Policy  |  "Which tenant gets to go next?"
+--------+---------+
         |
         v
+------------------+
| FIFO Ordering    |  "Which of this tenant's requests arrived first?"
+--------+---------+
         |
         v
    Dispatched to
    best replica
```

In [ ]:
# Enable the flow control feature gate in EPP configuration
import os

os.environ["NAMESPACE"] = "llm-d"

# The flow control feature gate is enabled via the EPP ConfigMap
# This patches the existing EPP config to turn on flow control

flow_control_patch = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-config
  namespace: llm-d
data:
  config.yaml: |
    featureGates:
      flowControl: true
    scoring:
      - name: prefix-cache-scorer
        weight: 0.7
      - name: load-aware-scorer
        weight: 0.3
    flowControl:
      enabled: true
      priorityHeader: "x-priority"
      tenantHeader: "x-tenant-id"
"""

# Write the patch file and apply it
with open("/tmp/epp-flow-control-patch.yaml", "w") as f:
    f.write(flow_control_patch)

!kubectl apply -f /tmp/epp-flow-control-patch.yaml

print("\nFlow control feature gate enabled.")
print("The EPP will now respect priority and fairness headers.")

In [ ]:
# Configure priority bands
# Priority bands map header values to numeric priorities.
# Higher numbers = higher priority = served first.

priority_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-priority-bands
  namespace: llm-d
data:
  priority-bands.yaml: |
    # Priority bands for the flow control system
    # Requests include an "x-priority" header with one of these values
    bands:
      - name: interactive
        priority: 100
        description: "Real-time user-facing requests (chat, search)"
      - name: batch
        priority: 50
        description: "Batch processing jobs (document analysis, embeddings)"
      - name: background
        priority: 10
        description: "Background tasks (pre-computation, cache warming)"
    defaultBand: batch  # Requests without the header get this priority
"""

with open("/tmp/epp-priority-bands.yaml", "w") as f:
    f.write(priority_config)

!kubectl apply -f /tmp/epp-priority-bands.yaml

print("Priority bands configured:")
print("  interactive: 100 (highest)")
print("  batch:        50 (default)")
print("  background:   10 (lowest)")

In [ ]:
# Configure fairness policy
# This sets up round-robin fairness across tenant IDs within the same priority band

fairness_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-fairness-policy
  namespace: llm-d
data:
  fairness-policy.yaml: |
    # Fairness policy: how to distribute capacity within a priority band
    policy: round-robin
    tenantHeader: "x-tenant-id"
    # round-robin: Each tenant gets an equal share of requests
    # weighted:    Tenants get shares proportional to their allocation
    # strict-fifo: No fairness, pure first-come-first-served
"""

with open("/tmp/epp-fairness-policy.yaml", "w") as f:
    f.write(fairness_config)

!kubectl apply -f /tmp/epp-fairness-policy.yaml

print("Fairness policy configured: round-robin across tenant IDs")

In [ ]:
# Deploy the updated router with flow control enabled
# Restart the EPP to pick up the new configuration

!kubectl rollout restart deployment/epp -n $NAMESPACE

print("Waiting for EPP to restart with flow control...")
!kubectl rollout status deployment/epp -n $NAMESPACE --timeout=120s

print("Router restarted with flow control enabled.")

# Verify the config was loaded
!kubectl logs deployment/epp -n $NAMESPACE --tail=20 | grep -i "flow\|priority\|fairness"

In [ ]:
# Simulate two tenants sending requests simultaneously
# Tenant A sends interactive requests; Tenant B sends batch requests
# We expect Tenant A's requests to be served first due to higher priority

import subprocess
import json
import time
import threading

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

results = {"tenant-a": [], "tenant-b": []}

def send_requests(tenant_id, priority, count=5):
    """Send requests as a specific tenant with a given priority."""
    for i in range(count):
        payload = json.dumps({
            "model": "Qwen/Qwen3-32B",
            "messages": [{"role": "user", "content": f"Request {i+1} from {tenant_id}"}],
            "max_tokens": 20,
        })
        start = time.time()
        proc = subprocess.run(
            ["curl", "-s", f"http://{GATEWAY_IP}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json",
             "-H", f"x-tenant-id: {tenant_id}",
             "-H", f"x-priority: {priority}",
             "-d", payload],
            capture_output=True, text=True
        )
        elapsed = time.time() - start
        results[tenant_id].append(elapsed)

# Run both tenants in parallel
print("Sending requests from two tenants simultaneously...")
print("  Tenant A: interactive priority (100)")
print("  Tenant B: batch priority (50)")
print()

t1 = threading.Thread(target=send_requests, args=("tenant-a", "interactive", 5))
t2 = threading.Thread(target=send_requests, args=("tenant-b", "batch", 5))

t1.start()
t2.start()
t1.join()
t2.join()

print("Results:")
print(f"  Tenant A (interactive) latencies: {[f'{l:.3f}s' for l in results['tenant-a']]}")
print(f"  Tenant B (batch) latencies:       {[f'{l:.3f}s' for l in results['tenant-b']]}")
print()

if results["tenant-a"] and results["tenant-b"]:
    import statistics
    avg_a = statistics.mean(results["tenant-a"])
    avg_b = statistics.mean(results["tenant-b"])
    print(f"  Tenant A avg: {avg_a:.3f}s")
    print(f"  Tenant B avg: {avg_b:.3f}s")
    print()
    print("Tenant A (interactive) should have lower average latency because")
    print("the flow control system prioritizes interactive requests.")

In [ ]:
# Observe fair distribution in EPP metrics
# The flow control system exposes per-tenant and per-priority metrics

import urllib.request

EPP_METRICS = "http://localhost:9090/metrics"  # Port-forward: kubectl port-forward svc/epp -n llm-d 9090:9090

try:
    with urllib.request.urlopen(EPP_METRICS, timeout=5) as resp:
        metrics = resp.read().decode("utf-8")

    print("=== Priority Band Metrics ===")
    for line in metrics.split("\n"):
        if "priority_band" in line and not line.startswith("#"):
            print(f"  {line}")

    print("\n=== Tenant Fairness Metrics ===")
    for line in metrics.split("\n"):
        if "tenant" in line and not line.startswith("#"):
            print(f"  {line}")

    print("\n=== Queue Depth by Priority ===")
    for line in metrics.split("\n"):
        if "queue_depth" in line and not line.startswith("#"):
            print(f"  {line}")

except Exception as e:
    print(f"Could not reach metrics endpoint: {e}")
    print("Tip: Port-forward the EPP metrics port:")
    print("  kubectl port-forward svc/epp -n llm-d 9090:9090")

## Late-Binding Scheduling and SLO Guardrails

llm-d uses **late-binding scheduling**, meaning it does not assign a request to
a specific replica until the replica actually has capacity to serve it. This is
different from early-binding systems that assign immediately and queue on the
backend.

### Why Late-Binding?

- **Better load distribution**: The EPP sees the real-time state of all replicas
  at the moment of dispatch, not a stale snapshot from when the request arrived.
- **Priority preemption**: A high-priority request that arrives while all replicas
  are busy can jump ahead of lower-priority requests in the central queue.
- **SLO awareness**: The EPP can reject or deprioritize requests that would
  violate SLO targets, rather than blindly queuing them.

### SLO Guardrails

You can configure SLO targets that the flow control system will try to honor:

```yaml
sloGuardrails:
  maxQueueWaitTime: 30s     # Drop/reject if queued longer than this
  maxTTFT: 5s               # Target time-to-first-token
  shedPolicy: reject        # What to do when SLO cannot be met
                            # Options: reject, deprioritize, log-only
```

When the system detects that a request's SLO cannot be met (e.g., queue is too
deep), it can reject the request with an HTTP 429, allowing the client to retry
or fall back to a different service.

## Summary and Production Recommendations

### Checklist for Production Flow Control

1. **Define priority bands** that match your business needs. Most deployments
   need 2-3 bands (interactive, batch, and optionally background).

2. **Choose a fairness policy**:
   - `round-robin` if all tenants should get equal capacity
   - `weighted` if some tenants have larger allocations
   - `strict-fifo` if you don't need fairness (single-tenant)

3. **Set the tenant header** consistently across all clients. Use API gateway
   middleware to inject the header if needed.

4. **Configure SLO guardrails** to prevent queue buildup during traffic spikes.
   Start with generous limits and tighten as you learn your traffic patterns.

5. **Monitor metrics** to verify that the flow control is working as expected:
   - `epp_priority_band_requests_total` - requests per band
   - `epp_tenant_requests_total` - requests per tenant
   - `epp_queue_wait_time_seconds` - time spent in the central queue
   - `epp_slo_violations_total` - SLO breaches (should be near zero)

### Common Pitfalls

- **Too many priority bands**: More than 3-4 bands adds complexity without
  clear benefit. Keep it simple.
- **Missing tenant headers**: Requests without the tenant header all land in
  a "default" bucket, defeating fairness. Enforce the header at the API gateway.
- **Overly strict SLOs**: Setting max queue wait time too low causes excessive
  rejections. Start at 30s and tune downward.
- **No monitoring**: Flow control is only useful if you can observe it. Set up
  dashboards and alerts on the EPP metrics before enabling flow control in
  production.